In [1]:
import torch
from torch import nn, optim
from torch.nn.functional import mse_loss
from tqdm import tqdm, trange

In [2]:
class MLP(nn.Module):
    def __init__(self, input_size: int, output_size: int, hidden_size: int, num_layers: int) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        for i in range(num_layers - 1):
            in_dim = input_size if i == 0 else hidden_size
            layers.append(nn.Linear(in_dim, hidden_size))
            layers.append(nn.LayerNorm(hidden_size))
            layers.append(nn.LeakyReLU())
        last_in = hidden_size if num_layers > 1 else input_size
        layers.append(nn.Linear(last_in, output_size))
        self._net = nn.Sequential(*layers)

    def forward(self, *inputs: torch.Tensor) -> torch.Tensor:
        return self._net(torch.cat(inputs, dim=-1))

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

state_dim = 32
message_dim = 32
action_dim = state_dim
hidden_dim = 128
num_layers = 3

def reward_fn(state: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
        return 5 - ((action - state / torch.pi) ** 2).mean(dim=-1, keepdim=True)

In [4]:
message_actor1, target_message_actor1 = (MLP(state_dim, message_dim, hidden_dim, num_layers).to(device) for _ in range(2))
final_actor, target_final_actor = (MLP(message_dim, action_dim, hidden_dim, num_layers).to(device) for _ in range(2))
message_critic1_h1, target_message_critic1_h1 = (MLP(state_dim + message_dim, 1, hidden_dim, num_layers).to(device) for _ in range(2))
message_critic1_h2, target_message_critic1_h2 = (MLP(state_dim + message_dim, 1, hidden_dim, num_layers).to(device) for _ in range(2))
final_critic_h1, target_final_critic_h1 = (MLP(state_dim + message_dim + action_dim, 1, hidden_dim, num_layers).to(device) for _ in range(2))
final_critic_h2, target_final_critic_h2 = (MLP(state_dim + message_dim + action_dim, 1, hidden_dim, num_layers).to(device) for _ in range(2))

online_models = [message_actor1, final_actor, message_critic1_h1, message_critic1_h2, final_critic_h1, final_critic_h2]
target_models = [target_message_actor1, target_final_actor, target_message_critic1_h1, target_message_critic1_h2, target_final_critic_h1, target_final_critic_h2]

for online, target in zip(online_models, target_models):
    target.load_state_dict(online.state_dict())
    for param in target.parameters():
        param.requires_grad_(False)

In [5]:
num_steps = 1000000
batch_size = 128
lr = 3e-4
tau = 0.001
sigma = 0.5
sigma_decay = 0.99999
sigma_min = 0.001

message_actor1_opt = optim.Adam(message_actor1.parameters(), lr=lr)
final_actor_opt = optim.Adam(final_actor.parameters(), lr=lr)
message_critic1_opt = optim.Adam(
    list(message_critic1_h1.parameters()) + list(message_critic1_h2.parameters()),
    lr=lr,
)
final_critic_opt = optim.Adam(
    list(final_critic_h1.parameters()) + list(final_critic_h2.parameters()),
    lr=lr,
)

def polyak_update(target: nn.Module, online: nn.Module, tau: float) -> None:
    for target_param, online_param in zip(target.parameters(), online.parameters()):
        target_param.data.copy_(tau * online_param.data + (1 - tau) * target_param.data)

In [ ]:
for step in trange(num_steps):
    for _ in range(2):
        with torch.no_grad():
            state = torch.randn(batch_size, state_dim, device=device)
            message1 = message_actor1(state)
            message1 = message1 + torch.randn_like(message1) * sigma
            action = target_final_actor(message1 * 0)
            reward = reward_fn(state, action)
        ma1_q1 = message_critic1_h1(state, message1 * 0)
        ma1_q2 = message_critic1_h2(state, message1 * 0)
        fc_q1 = target_final_critic_h1(state, message1 * 0, action)
        fc_q2 = target_final_critic_h2(state, message1 * 0, action)
        fc_q = torch.minimum(fc_q1, fc_q2)
        mc1_loss = mse_loss(ma1_q1, fc_q) + mse_loss(ma1_q2, fc_q)
        message_critic1_opt.zero_grad()
        mc1_loss.backward()
        message_critic1_opt.step()

        with torch.no_grad():
            state = torch.randn(batch_size, state_dim, device=device)
            message1 = target_message_actor1(state)
            message1 = message1 + torch.randn_like(message1) * sigma
            action = final_actor(message1 * 0)
            reward = reward_fn(state, action)
        fc_q1 = final_critic_h1(state, message1 * 0, action)
        fc_q2 = final_critic_h2(state, message1 * 0, action)
        fc_loss = mse_loss(fc_q1, reward) + mse_loss(fc_q2, reward)
        final_critic_opt.zero_grad()
        fc_loss.backward()
        final_critic_opt.step()

    with torch.no_grad():
        state = torch.randn(batch_size, state_dim, device=device)
    message1 = message_actor1(state)
    action = target_final_actor(message1 * 0)
    mc1_q1 = target_message_critic1_h1(state, message1 * 0)
    mc1_q2 = target_message_critic1_h2(state, message1 * 0)
    mc1_q = torch.minimum(mc1_q1, mc1_q2)
    ma1_loss = -mc1_q.mean()
    message_actor1_opt.zero_grad()
    ma1_loss.backward()
    message_actor1_opt.step()

    with torch.no_grad():
        state = torch.randn(batch_size, state_dim, device=device)
        message1 = target_message_actor1(state)
        message1 = message1 + torch.randn_like(message1) * sigma
    action = final_actor(message1 * 0)
    reward = reward_fn(state, action)
    fc_q1 = target_final_critic_h1(state, message1 * 0, action)
    fc_q2 = target_final_critic_h2(state, message1 * 0, action)
    fc_q = torch.minimum(fc_q1, fc_q2)
    fa_loss = -fc_q.mean()
    final_actor_opt.zero_grad()
    fa_loss.backward()
    final_actor_opt.step()

    polyak_update(target_message_actor1, message_actor1, tau)
    polyak_update(target_final_actor, final_actor, tau)
    polyak_update(target_message_critic1_h1, message_critic1_h1, tau)
    polyak_update(target_message_critic1_h2, message_critic1_h2, tau)
    polyak_update(target_final_critic_h1, final_critic_h1, tau)
    polyak_update(target_final_critic_h2, final_critic_h2, tau)

    sigma = max(sigma * sigma_decay, sigma_min)

    if step % 100 == 0:
        print(f"Step {step}, sigma: {sigma:.6f}, ma1l: {ma1_loss.item():.6f}, fal: {fa_loss.item():.6f}, mc1l: {mc1_loss.item():.6f}, fcl: {fc_loss.item():.6f}, rw: {reward.mean().item():.6f}")

In [58]:
no_message = torch.zeros(batch_size, message_dim).to(device)

for step in range(num_steps):
    with torch.no_grad():
        state = torch.rand(batch_size, state_dim).to(device)
        action = final_actor(state)
        action = action + torch.randn_like(action) * sigma
        reward = reward_fn(state, action.detach())

    # Critic update
    q1 = final_critic_h1(state, no_message, action.detach())
    q2 = final_critic_h2(state, no_message, action.detach())
    critic_loss = mse_loss(q1, reward) + mse_loss(q2, reward)

    final_critic_opt.zero_grad()
    critic_loss.backward()
    final_critic_opt.step()
    polyak_update(target_final_critic_h1, final_critic_h1, tau)
    polyak_update(target_final_critic_h2, final_critic_h2, tau)

    if step % 20 == 0:
        # Actor update
        state = torch.rand(batch_size, state_dim).to(device)
        action = final_actor(state)
        q1 = target_final_critic_h1(state, no_message, action)
        q2 = target_final_critic_h2(state, no_message, action)
        q = torch.minimum(q1, q2)
        actor_loss = -q.mean()

        final_actor_opt.zero_grad()
        actor_loss.backward()
        final_actor_opt.step()
        polyak_update(target_final_actor, final_actor, tau)

    sigma = max(sigma * sigma_decay, sigma_min)

    if step % 200 == 0:
        print(f"Step {step} | critic_loss: {critic_loss.item():.4f} | actor_loss: {actor_loss.item():.4f} | reward: {reward.mean().item():.4f} | sigma: {sigma:.4f}")

Step 0 | critic_loss: 0.0007 | actor_loss: -5.0031 | reward: 4.7522 | sigma: 0.5000
Step 200 | critic_loss: 0.0001 | actor_loss: -5.0067 | reward: 4.7428 | sigma: 0.4990
Step 400 | critic_loss: 0.0001 | actor_loss: -5.0107 | reward: 4.7479 | sigma: 0.4980
Step 600 | critic_loss: 0.0001 | actor_loss: -5.0129 | reward: 4.7453 | sigma: 0.4970
Step 800 | critic_loss: 0.0001 | actor_loss: -5.0144 | reward: 4.7503 | sigma: 0.4960
Step 1000 | critic_loss: 0.0001 | actor_loss: -5.0155 | reward: 4.7583 | sigma: 0.4950
Step 1200 | critic_loss: 0.0000 | actor_loss: -5.0158 | reward: 4.7465 | sigma: 0.4940
Step 1400 | critic_loss: 0.0000 | actor_loss: -5.0171 | reward: 4.7571 | sigma: 0.4930
Step 1600 | critic_loss: 0.0000 | actor_loss: -5.0172 | reward: 4.7637 | sigma: 0.4921
Step 1800 | critic_loss: 0.0001 | actor_loss: -5.0174 | reward: 4.7670 | sigma: 0.4911
Step 2000 | critic_loss: 0.0001 | actor_loss: -5.0164 | reward: 4.7621 | sigma: 0.4901
Step 2200 | critic_loss: 0.0000 | actor_loss: -5.0

KeyboardInterrupt: 

In [ ]:
no_message = torch.zeros(batch_size, message_dim).to(device)

for step in range(num_steps):
    with torch.no_grad():
        state = torch.rand(batch_size, state_dim).to(device)
        action = final_actor(state)
        action = action + torch.randn_like(action) * sigma
        reward = reward_fn(state, action.detach())

    # Critic update
    q1 = final_critic_h1(state, no_message, action.detach())
    q2 = final_critic_h2(state, no_message, action.detach())
    critic_loss = mse_loss(q1, reward) + mse_loss(q2, reward)

    final_critic_opt.zero_grad()
    critic_loss.backward()
    final_critic_opt.step()
    polyak_update(target_final_critic_h1, final_critic_h1, tau)
    polyak_update(target_final_critic_h2, final_critic_h2, tau)

    if step % 20 == 0:
        # Actor update
        state = torch.rand(batch_size, state_dim).to(device)
        action = final_actor(state)
        q1 = target_final_critic_h1(state, no_message, action)
        q2 = target_final_critic_h2(state, no_message, action)
        q = torch.minimum(q1, q2)
        actor_loss = -q.mean()

        final_actor_opt.zero_grad()
        actor_loss.backward()
        final_actor_opt.step()
        polyak_update(target_final_actor, final_actor, tau)

    sigma = max(sigma * sigma_decay, sigma_min)

    if step % 200 == 0:
        print(f"Step {step} | critic_loss: {critic_loss.item():.4f} | actor_loss: {actor_loss.item():.4f} | reward: {reward.mean().item():.4f} | sigma: {sigma:.4f}")